# Ames 주택 가격 실습 — 예측이 놓친 차이 찾아보기

**같은 모델에 정보를 다르게 전달하면 주택 가격 예측이 달라질까요?**

함께하기에서 다룬 회귀를 새로운 주택 데이터에 적용합니다. 첫 예측의 오차를 살펴보고, 데이터에서 발견한 차이를 모델에 어떻게 전달할지 직접 생각해 봅니다.

**진행 순서:** 준비 → 첫 예측 → 관심 오차 선택 → 기록과 관계 탐색 → 개선 가설 → 결과 비교

- **함께 진행:** Notion 요청 예시로 코드를 받아 실행하고 결과를 읽습니다.
- **직접 탐색:** 관찰한 결과를 바탕으로 바꿀 방향을 정하고 Continue에 자신의 말로 요청합니다.
- 코드를 외울 필요는 없습니다. 위에서부터 실행하고, 코드가 필요한 곳의 빈 셀에 붙여넣으세요. 성능이 좋아져야만 성공한 실험은 아닙니다.

## 1. 어떤 데이터인가요?

미국 아이오와주 Ames에서 2006~2010년에 거래된 주택 자료입니다. **한 행은 주택 한 건의 거래 기록**이며, 예측 대상은 당시 거래 가격입니다. 오늘날의 실제 매매 가격을 예측하는 자료는 아닙니다.

원본 2,930행에서 수업에 사용할 입력 12개와 가격을 선택합니다.

| 항목 | 의미 | 단위·범위 |
|---|---|---|
| Gr Liv Area | 지상 주거 면적 | 제곱피트(ft²) |
| Lot Area | 대지 면적 | ft² |
| Overall Qual | 자재와 마감의 전체 품질 등급 | 1~10, 높을수록 좋음 |
| Overall Cond | 주택의 전체 상태 등급 | 1~10 척도, 높을수록 좋음 |
| Year Built | 건축 연도 | 연도 |
| Year Remod/Add | 리모델링·증축 연도 | 변경이 없으면 건축 연도와 같음 |
| Total Bsmt SF | 지하실 전체 면적 | ft² |
| Garage Area | 차고 면적 | ft² |
| Garage Cars | 차고에 주차할 수 있는 차량 수 | 대 |
| TotRms AbvGrd | 지상층 방 수, 욕실 제외 | 개 |
| Full Bath | 지상층의 완전한 욕실 수 | 개 |
| Yr Sold | 거래 연도 | 연도 |
| SalePrice | 실제 거래 가격, 예측 대상 | 미국 달러 |

1ft²는 약 0.093m²입니다. `Overall Qual`은 자재·마감의 품질, `Overall Cond`는 집의 전반적인 상태를 평가한 값입니다. 두 항목은 서로 다릅니다.

출처: [원저자 논문](https://jse.amstat.org/v19n3/decock.pdf) · [데이터 설명](https://jse.amstat.org/v19n3/decock/DataDocumentation.txt)

## 2. 실습 준비

### 함께 준비 · 도구와 데이터

표·그래프와 모델에 필요한 도구, 한글 글꼴을 준비합니다. 설정 코드는 외울 필요가 없습니다.

노트북과 함께 제공된 `data/AmesHousing.txt`를 읽습니다. 데이터 파일은 이미 내려받아 두었으므로 이 셀은 인터넷 연결 없이 실행할 수 있습니다. 원본은 `df_raw`, 선택한 항목의 작업용 표는 `df`에 보관합니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_style("whitegrid")
available_fonts = {font.name for font in fm.fontManager.ttflist}
korean_font = next((name for name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]
                    if name in available_fonts), None)
if korean_font:
    plt.rcParams["font.family"] = korean_font
else:
    print("한글 글꼴을 찾지 못했습니다.")
plt.rcParams["axes.unicode_minus"] = False

# 노트북 폴더 또는 프로젝트 루트에서 실행할 수 있습니다.
data_path = Path("data/AmesHousing.txt")
if not data_path.exists():
    data_path = Path("3. 회귀/data/AmesHousing.txt")
df_raw = pd.read_csv(data_path, sep="\t")
features = ["Gr Liv Area", "Lot Area", "Overall Qual", "Overall Cond",
            "Year Built", "Year Remod/Add", "Total Bsmt SF", "Garage Area",
            "Garage Cars", "TotRms AbvGrd", "Full Bath", "Yr Sold"]
df = df_raw[features + ["SalePrice"]].copy()
print(f"거래 기록: {len(df):,}행 / 입력: {len(features)}개")
display(df.head())

### 함께 준비 · 빈 값이 있는 기록 확인

결측치가 어느 항목에 몇 개 있는지 확인하고 해당 행도 직접 봅니다. 차고 면적이 0인 것과 차고 면적을 알 수 없는 결측치는 다릅니다.

In [ ]:
display(df.isna().sum().rename("결측치 수"))
display(df.loc[df.isna().any(axis=1)])

### 함께 준비 · 나눈 뒤 빈 값 채우기

행을 제외하면 그 주택의 다른 정보도 사라집니다. 이번에는 기록을 유지하고 입력의 빈 값을 **훈련 자료에서 구한 중앙값**으로 채웁니다. 이는 실제 값을 복원하는 것이 아니라 이번 비교를 위한 간단한 처리입니다.

80%로 학습하고 20%로 예측을 비교합니다. 이후에도 같은 평가 기록을 사용합니다. 이 20%는 실습 중 개선 비교에 반복 사용하므로, 최종 성능을 확인하려면 개선 과정에서 보지 않은 별도 자료가 필요합니다.

In [ ]:
X = df[features].copy()
y = df["SalePrice"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
before_train = X_train.copy()
before_test = X_test.copy()
fill_values = X_train.median()
X_train = X_train.fillna(fill_values)
X_test = X_test.fillna(fill_values)
missing_cols = X.columns[X.isna().any()]

## 3. Continue로 분석 시작하기

[Notion 요청 예시](https://app.notion.com/p/regression_practice-ipynb-3dbb39501c3880de8d6af7fd342825bb)의 **3. 처음 한 번 전달할 지침**과 데이터 설명·준비 코드를 Continue에 전달하세요.

같은 대화에서 지금 단계의 코드만 요청합니다. **산점도를 본 다음에는 관심 있는 오차와 분석 항목을 직접 고릅니다.** Notion은 함께 준비할 때만 참고하고, 직접 탐색은 자신의 말로 요청하세요.

오류가 나면 실행한 코드와 오류 메시지를, 해석을 물을 때는 실제 표·그래프를 함께 전달하세요. AI가 노트북 출력을 자동으로 알고 있는 것은 아닙니다.

## 4. 첫 예측은 어느 정도일까요?

### 함께 진행 · 기준과 선형회귀

Notion의 **4-1. 기준과 선형회귀** 요청 예시를 사용하세요. 훈련 가격의 중앙값 하나로 예측한 결과와, 12개 입력을 사용한 선형회귀를 비교합니다.

- **MAE:** 실제와 예측의 절대 차이를 평균 낸 값입니다. 20,000달러라면 평균적인 절대 차이가 그만큼이라는 뜻이며, 모든 예측이 그 범위에 있다는 뜻은 아닙니다.
- **RMSE:** 큰 오차에 더 민감한 지표입니다. MAE와 같은 달러 단위이고 작을수록 좋습니다.
- **±20% 이내 비율:** 실제 100,000달러인 집이라면 80,000~120,000달러로 예측한 경우입니다. 높을수록 범위 안에 들어온 기록이 많습니다. 20%는 수업용 가정입니다.

중앙값 기준은 MAE의 비교 기준입니다. 선형회귀는 학습할 때 오차의 제곱합을 줄이지만, 수업에서는 MAE를 대표 평가 지표로 읽습니다.

### 함께 진행 · 실제·예측 산점도

Notion의 **4-2. 실제·예측 산점도** 요청 예시를 사용하세요. 점 하나는 평가용 주택 한 건입니다.

가로축은 실제 가격, 세로축은 예측 가격입니다. 대각선에 가까울수록 잘 맞고, 위쪽은 높게 예측, 아래쪽은 낮게 예측입니다. 두 축은 달러 단위입니다. 음수나 큰 예측을 숨기지 않고 확인합니다.

## 5. 어떤 오차를 더 살펴볼까요?

### 함께 진행 · 점을 기록으로 확인하기

Notion의 **5. 예측 결과표** 요청 예시로 입력 항목·실제 가격·예측 가격·오차를 한 표에 모읍니다.

**오차 = 예측−실제**입니다. 양수면 높게 예측, 음수면 낮게 예측한 것입니다. 절대 오차는 방향을 빼고 얼마나 틀렸는지를 나타냅니다.

### 직접 탐색 · 관심 있는 기록 선택

앞의 산점도를 보고 아래 중 **하나를 골라** 해당 기록을 확인하는 코드를 직접 요청하세요.

- **높게 예측한 집:** 대각선 위에 멀리 있는 점. 양의 오차가 큰 기록의 정보표
- **낮게 예측한 집:** 대각선 아래에 멀리 있는 점. 음의 오차가 큰 기록의 정보표
- **크게 틀린 집:** 방향과 관계없이 대각선에서 먼 점. 절대 오차가 큰 기록의 정보표

처음에는 10건 정도를 살펴보세요. 면적·연식·품질·차고 등에서 눈에 띄는 항목을 하나 고릅니다. 비슷한 실제 가격대에서 잘 맞춘 기록도 함께 보면 차이를 비교하기 쉽습니다. 원본과 평가 기록은 삭제하지 않습니다.

## 6. 눈에 띈 항목과 오차의 관계 확인하기

### 직접 탐색 · 표와 그래프 고르기

5절에서 관심이 생긴 항목 하나를 정하고, 아래에서 맞는 방법을 골라 직접 요청하세요. 선택한 몇 건에서 보인 차이가 **전체 평가 기록에서도 나타나는지** 확인합니다.

| 궁금한 점 | 확인할 표·그래프 |
|---|---|
| 면적이나 건축 연도에 따라 오차가 달라질까? | 가로축은 선택 항목, 세로축은 오차인 산점도와 0 기준선 |
| 등급이나 종류에 따라 다르게 틀릴까? | 그룹별 건수·MAE·평균 오차 표와 막대그래프 |
| 어떤 시설이 있는 집과 없는 집이 다를까? | 유무별 건수·오차 표 또는 오차 분포 상자그림 |

평균 오차는 양수·음수가 상쇄될 수 있으므로 MAE도 함께 봅니다. 기록이 적은 그룹은 소수의 큰 오차에 영향을 받기 쉽습니다.

관계가 보인다면 **훈련 자료**에서도 해당 항목의 분포나 실제 가격과의 관계를 확인해보세요. 큰 오차 기록에 공통점이 있다는 것만으로 오류나 원인이 확인된 것은 아닙니다.

## 7. 관찰한 차이를 바탕으로 한 가지 바꾸기

### 직접 탐색 · 개선 가설과 실험

**어떤 차이를 발견했고, 어떤 정보를 다르게 전달하면 도움이 될까요?** 바꾸고 싶은 이유를 먼저 짧게 말한 뒤 Continue에 요청하세요.

막막하면 다음 방법 중 관찰한 내용에 맞는 것을 고르세요. 반드시 이 중에서 골라야 하는 것은 아닙니다.

| 바꿀 수 있는 방법 | 생각해볼 점 |
|---|---|
| 숫자를 구간이나 종류로 구분하기 | 값이 일정하게 증가하는 것보다 구분해서 전달하는 편이 맞을까? |
| 유무를 새 정보로 만들기 | 얼마나 큰지와 별개로 있는지 없는지가 중요할까? |
| 관련된 두 항목으로 비율 만들기 | 전체 크기보다 상대적인 크기가 더 잘 설명할까? |

**한 번에 한 가지 표현만 바꿉니다.** 모델은 선형회귀, 정답은 원래 달러 가격으로 유지하고, 같은 평가 기록으로 비교합니다. 가격 자체를 입력에 넣지는 않습니다.

원본 입력은 보존하고 새 복사본에서 바꾸세요. 구간 경계 등 데이터로 정하는 기준은 훈련 자료에서 정해 평가 입력에도 똑같이 적용합니다. 비율을 만든다면 분모가 0인 기록의 처리도 확인하세요.

새 모델은 `model_changed`, 새 예측은 `pred_changed`로 보관하도록 요청합니다. 나머지 구현은 AI의 도움을 받아도 됩니다. 모든 방법이 좋아질 필요는 없습니다.

## 8. 처음 발견한 문제가 줄었나요?

### 함께 진행 · 전체 결과 비교

Notion의 **8. 변경 전후 비교** 요청 예시로 MAE·RMSE·±20% 이내 비율과 실제·예측 산점도를 비교합니다.

MAE·RMSE는 작을수록, 허용 범위 이내 비율은 클수록 좋습니다. 지표가 서로 다른 방향으로 움직일 수 있습니다.

### 직접 탐색 · 관심 기록과 그룹 다시 보기

5절에서 골랐던 **같은 기록**, 6절에서 살펴본 **같은 그룹이나 구간**의 변경 전후 오차를 직접 비교하세요. 변경 후 잘 맞는 기록만 새로 고르면 공정한 비교가 어렵습니다.

처음 발견한 문제가 줄었는지, 다른 기록에서는 오차가 늘지 않았는지 살펴봅니다. 전체 평균이 좋아져도 모든 집에서 좋아진 것은 아닙니다.

## 9. 궁금한 내용 한 가지 더 확인하기

아래 중 **하나를 선택**해 직접 요청하세요.

- **같은 시도를 조금 다르게:** 구간을 바꾸거나 새 정보를 추가할지 대체할지 비교합니다. 한 번에 바꾸는 것은 하나로 제한합니다.
- **다른 항목에서도 확인:** 6절에서 눈에 띈 다른 항목에 같은 분석 방법을 적용합니다. 처음부터 새로 모델을 만들 필요는 없습니다.
- **아직 크게 틀리는 기록 확인:** 변경 후에도 오차가 큰 집의 정보표와 관심 항목의 산점도를 보고 다음에 확인할 점을 찾습니다.

새 모델을 만들었다면 같은 평가 기록에서 비교합니다. 그렇지 않다면 추가로 관찰한 내용을 표·그래프로 보여주면 됩니다.

## 10. 결과 공유

표나 그래프 하나를 보여주며 **고른 오차 → 발견한 공통점 → 바꾼 정보 → 실제 결과**를 짧게 설명하세요.

좋아지지 않았다면 예상과 결과가 어떻게 달랐는지 이야기하면 됩니다. AI에게 해석을 물을 때는 실제 출력을 함께 전달하고 설명과 맞는지 확인하세요.

## 선택 심화 · 비교 기준을 더 확인하기

시간이 남으면 둘 중 하나를 골라 직접 요청하세요.

- **허용 범위:** 같은 예측에서 ±10%, ±20%, ±30% 이내 비율 비교. 허용 범위를 넓혀 비율이 높아져도 모델이 개선된 것은 아닙니다.
- **교차검증:** 훈련 자료 안에서만 같은 5개 분할로 기본 모델과 변경 모델 비교. 결측치를 채울 중앙값과 데이터로 정하는 묶음 기준은 각 분할의 학습 부분에서 다시 구합니다. 이를 위해 준비 코드에서 보관한 `before_train`과 `y_train`을 사용하고 기존 평가 자료는 사용하지 않습니다. 분할마다 결과가 비슷한지 확인하세요.